In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "numpy", "matplotlib", "seaborn"])
print("Packages installed successfully!")

Packages installed successfully!


In [2]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# cleaning trailing spaces
df.columns = df.columns.str.strip()

# verify it was cleaned
print([col in df.columns if 'Delay' in col else False for col in df.columns])

NameError: name 'df' is not defined

In [ ]:
all_files = glob.glob("*.csv")

use_cols = [
    'FlightDate', 'Reporting_Airline', 'Origin', 'DEST', 
    'CRSDepTime','DEP_DELAY', 'CRSArrTime', 'ARR_DELAY',
    'TaxiOut', 'TaxiIn', 'AirTime', 'Cancelled', 
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay'
    ]

df_list = []
for file in all_files:
    temp_df = pd.read_csv(file, usecols=lambda c: c in use_cols)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)
print(f"Combined datasets across all months! Total rows: {len(df)}")

In [ ]:
# 1. Print all available columns to see what we are working with
print("All columns currently in df:", df.columns.tolist())

# 2. Filter out cancelled flights and clean CRSDepTime
active_flights = df[df['Cancelled'] == 0].copy()
active_flights['CRSDepTime_clean'] = active_flights['CRSDepTime'].astype(str).str.split('.').str[0].str.zfill(4)
active_flights['DepHour'] = pd.to_numeric(active_flights['CRSDepTime_clean'].str[:2], errors='coerce')

# 3. Choose your hub
target_hub = 'DFW'
hub_flights = active_flights[active_flights['Origin'] == target_hub].copy()
print(f"\nTotal active departures from {target_hub}: {len(hub_flights):,}")

In [ ]:
#Extract Departure hours and Calculate queue wait times
df.columns = df.columns.str.strip()

active_flights  = df[df['Cancelled'] == 0].copy()

active_flights['CRSDepTime_clean'] = active_flights['CRSDepTime'].astype(str).str.split('.').str[0].str.zfill(4)
active_flights['DepHour'] = pd.to_numeric(active_flights['CRSDepTime_clean'].str[:2], errors='coerce')

#Flight hubs chosen based on size and number of flights (Busiest airports in the US)
target_hub = 'DFW' # Dallas Fort Worth International Airport
hub_flights = active_flights[active_flights['Origin'] == target_hub].copy()
print(f"Total active departures from {target_hub}: {len(hub_flights)}")

hub_flights['TaxiOut_clean'] = pd.to_numeric(hub_flights['TaxiOut'], errors='coerce').fillna(0)

# Get Arrival rate (lambda) and average wait time (W)
hourly_wait = hub_flights.groupby('DepHour').agg(
    ArrivalRate = ('FlightDate', 'count'), # Lambda
    AvgWaitTime = ('TaxiOut_clean', 'mean') # W
).reset_index()

print("Hourly queueing summary calculated for flights departing from the hub.")
display(hourly_wait.head(10))

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax1 = plt.subplots(figsize=(12, 6))

sns.lineplot(
    data=hourly_wait, 
    x='DepHour', 
    y='AvgWaitTime', 
    marker = 'o',
    color = 'crimson',
    ax=ax1,
    linewidth=2.5,
    label='Average Runway Wait Time (minutes)'
)

ax1.set_title(f'Flight Volume vs. Runway Wait Time for Departures from {target_hub}', fontsize=14,
    fontweight='bold', pad=15)
ax1.set_xlabel('Scheduled Departure Hour (Local Time)', fontsize=12)
ax1.set_ylabel('Average Runway Wait Time ($W$ in minutes)', color='crimson', fontsize=12)
ax1.tick_params(axis='y', labelcolor='crimson')

ax2 = ax1.twinx()
sns.barplot(
    data=hourly_wait, 
    x='DepHour', 
    y='ArrivalRate', 
    alpha=0.4, 
    ax=ax2, 
    color='#1B4F72', 
    label='Flight Volume (Arrival Rate $\lambda$)'
)

ax2.set_ylabel('Total Flight Departures', color='#1B4F72', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#1B4F72')
ax2.grid(False)
ax1.grid(False)

plt.xticks(range(0, 24))
fig.tight_layout()
plt.show()

In [ ]:
peak_hour_row = hourly_wait.loc[hourly_wait['ArrivalRate'].idxmax()]
lambda_peak = peak_hour_row['ArrivalRate']
peak_dep_hour = peak_hour_row['DepHour']
avg_wait_peak = peak_hour_row['AvgWaitTime']

print(f"Peak Departure Hour at DFW: {int(peak_dep_hour)}:00")
print(f"Peak Arrival Rate (lambda): {lambda_peak:,} flights/hour")
print(f"Average Wait Time at Peak: {avg_wait_peak:.2f} minutes")

c_runways = 3

runway_capacity_per_hour = 40 # standard FAA optimal throughput per runway
total_system_capacity = c_runways * runway_capacity_per_hour

rho_peak = lambda_peak / total_system_capacity
print(f"\nEstimated Peak System Capacity ({c_runways} runways @ {runway_capacity_per_hour}/hr): {total_system_capacity} departures/hr")
print(f"Calculated Traffic Intensity (Rho): {rho_peak:.2f} (Note: > 1.0 indicates oversaturated queue)")

In [ ]:
target_hub = 'ORD' # Chicago O'Hare International Airport

print(f"\nNow analyzing departures from {target_hub}...")

hub_flights = active_flights[active_flights['Origin'] == target_hub].copy()
print(f"Total active departures from {target_hub}: {len(hub_flights)}")

hub_flights['TaxiOut_clean'] = pd.to_numeric(hub_flights['TaxiOut'], errors='coerce').fillna(0)

# Get Arrival rate (lambda) and average wait time (W)
hourly_wait = hub_flights.groupby('DepHour').agg(
    ArrivalRate = ('FlightDate', 'count'), # Lambda
    AvgWaitTime = ('TaxiOut_clean', 'mean') # W
).reset_index()

sns.set_theme(style="white")
fig, ax1 = plt.subplots(figsize=(12, 6))

sns.lineplot(
    data=hourly_wait, 
    x='DepHour', 
    y='AvgWaitTime', 
    marker = 'o',
    color = 'crimson',
    ax=ax1,
    linewidth=2.5,
    label='Average Runway Wait Time (minutes)'
)

ax1.set_title(f'Airport Congestion Curve: Arrival Rate vs. Runway Wait Times ({target_hub})', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Scheduled Departure Hour (Local)', fontsize=12)
ax1.set_ylabel('Average Wait Time ($W$ in minutes)', fontsize=12, color='crimson')
ax1.tick_params(axis='y', labelcolor='crimson')
ax1.grid(False)

ax2 = ax1.twinx()
sns.barplot(
    data=hourly_wait, 
    x='DepHour', 
    y='ArrivalRate', 
    alpha=0.4, 
    color='#1B4F72', 
    ax=ax2, 
    label='Flight Volume (Arrival Rate $\lambda$)'
)
ax2.set_ylabel('Total Flight Departures', fontsize=12, color='#1B4F72')
ax2.tick_params(axis='y', labelcolor='#1B4F72')
ax2.grid(False)

plt.xticks(range(0, 24))
fig.tight_layout()
plt.show()

In [ ]:
# DFW summary generated
hub_flights_dfw = active_flights[active_flights['Origin'] == 'DFW'].copy()
hub_flights_dfw['TaxiOut_clean'] = pd.to_numeric(hub_flights_dfw['TaxiOut'], errors='coerce').fillna(0)
hourly_wait_dfw = hub_flights_dfw.groupby('DepHour').agg(
    ArrivalRate=('FlightDate', 'count'),
    AvgWaitTime=('TaxiOut_clean', 'mean')
).reset_index()
hourly_wait_dfw['Hub'] = 'DFW'

# ORD summary generated
hub_flights_ord = active_flights[active_flights['Origin'] == 'ORD'].copy()
hub_flights_ord['TaxiOut_clean'] = pd.to_numeric(hub_flights_ord['TaxiOut'], errors='coerce').fillna(0)
hourly_wait_ord = hub_flights_ord.groupby('DepHour').agg(
    ArrivalRate=('FlightDate', 'count'),
    AvgWaitTime=('TaxiOut_clean', 'mean')
).reset_index()
hourly_wait_ord['Hub'] = 'ORD'

display(hourly_wait_dfw.head(10))
display(hourly_wait_ord.head(10))

In [ ]:
# Summary dataframe for both hubs
combined_hubs = pd.concat([hourly_wait_dfw, hourly_wait_ord])

# Direct comparison of average wait times between DFW and ORD
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(
    data=combined_hubs, 
    x='DepHour', 
    y='AvgWaitTime', 
    hue='Hub', 
    marker='o', 
    markersize=8,
    linewidth=3,
    palette={'DFW': '#D35400', 'ORD': '#2980B9'} # DFW in dark orange, ORD in deep blue
)

sns.despine(left=False, bottom=False)
ax.legend(frameon=False, title='Airport Hub', fontsize=11, title_fontsize=12)

ax.set_title('Runway Queue Comparison: DFW vs. ORD (Average Wait Times)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Scheduled Departure Hour (Local)', fontsize=12)
ax.set_ylabel('Average Runway Wait Time ($W$ in minutes)', fontsize=12)
ax.grid(False)

plt.xticks(range(0, 24))
fig.tight_layout()
plt.show()

In [ ]:
num_days = active_flights['FlightDate'].nunique() # Total number of days in the dataset

runway_capacity_per_hour = 40 # Standard FAA optimal benchmark per runway (~40 operations/hr)

hub_configs = {
    'DFW': {'df': hourly_wait_dfw, 'runways': 3},
    'ORD': {'df': hourly_wait_ord, 'runways': 3} # Or 4 depending on configuration
}

print("=" * 45)
print("Peak Queue Utilization Metrics (rho = \lambda / c*mu$)")
print("=" * 45)

for hub_name, config in hub_configs.items():
    df = config['df']
    runways = config['runways']
    
    # Find peak hour row
    peak_row = df.loc[df['ArrivalRate'].idxmax()]
    lambda_total = peak_row['ArrivalRate']
    lambda_peak = lambda_total / num_days # true average arrival rate per hour
    avg_wait_peak = peak_row['AvgWaitTime']
    
    total_capacity = runways * runway_capacity_per_hour
    rho = lambda_peak / total_capacity
    
    print(f"[{hub_name}]")
    print(f"  - Active Departure Runways ($c$): {runways}")
    print(f"  - Peak Arrival Rate ($\lambda$): {lambda_peak:.1f} flights/hr (Hour {int(peak_row['DepHour'])}:00)")
    print(f"  - Peak Wait Time ($W$): {avg_wait_peak:.2f} minutes")
    print(f"  - Traffic Intensity (rho$): {rho:.2f}")
    
    if rho >= 1.0:
        print(f"  - Status: Oversaturated (rho >= 1.0$) — Queue grows exponentially.")
    else:
        print(f"  - Status: Sub-critical queue (Operating under theoretical capacity).")
    print("-" * 45)

In [ ]:
# ATL summary generated
hub_flights_atl = active_flights[active_flights['Origin'] == 'ATL'].copy()
hub_flights_atl['TaxiOut_clean'] = pd.to_numeric(hub_flights_atl['TaxiOut'], errors='coerce').fillna(0)
hourly_wait_atl = hub_flights_atl.groupby('DepHour').agg(
    ArrivalRate=('FlightDate', 'count'),
    AvgWaitTime=('TaxiOut_clean', 'mean')
).reset_index()
hourly_wait_atl['Hub'] = 'ATL'

# combined dataframe for all three hubs
combined_three_hubs = pd.concat([hourly_wait_dfw, hourly_wait_ord, hourly_wait_atl])

sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(
    data=combined_three_hubs, 
    x='DepHour', 
    y='AvgWaitTime', 
    hue='Hub', 
    marker='o', 
    markersize=8,
    linewidth=3,
    palette={
        'DFW': '#D35400', # Orange
        'ORD': '#1B4F72', # Blue
        'ATL': '#27AE60'  # Green
    }
)

ax.set_title('Runway Queue Comparison: DFW vs. ORD vs. ATL (Average Wait Times)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Scheduled Departure Hour (Local)', fontsize=12, labelpad=10)
ax.set_ylabel('Average Runway Wait Time ($W$ in minutes)', fontsize=12, labelpad=10)
ax.legend(frameon=False, title='Hub Airport', fontsize=11, title_fontsize=12)
ax.grid(False)

plt.xticks(range(0, 24))
fig.tight_layout()
plt.show()

# printing peak queue utilization metrics for all three hubs
print("=" * 45)
print("Peak Queue Utilization Metrics (rho = \lambda / c*mu$)")
print("=" * 45)

hub_data_dict = {'DFW': (hourly_wait_dfw, 3), 'ORD': (hourly_wait_ord, 3), 'ATL': (hourly_wait_atl, 4)}
runway_capacity_per_hour = 40 # Standard FAA benchmark

for hub_name, (df, runways) in hub_data_dict.items():
    peak_row = df.loc[df['ArrivalRate'].idxmax()]
    lambda_peak = peak_row['ArrivalRate']
    total_capacity = runways * runway_capacity_per_hour
    rho = lambda_peak / total_capacity
    
    print(f"[{hub_name}]")
    print(f"  - Active Departure Runways ($c$): {runways}")
    print(f"  - Peak Arrival Rate ($\lambda$): {lambda_peak} flights/hr at hour {int(peak_row['DepHour'])}")
    print(f"  - Traffic Intensity (rho$): {rho:.2f}")
    if rho >= 1.0:
        print(f"  - Status: Oversaturated (rho > 1.0$) — Queue will grow exponentially.")
    else:
        print(f"  - Status: Stable sub-critical queue.")
    print("-" * 45)

In [ ]:
# compiled summary statistics for the three hubs
summary_data = []
num_days = active_flights['FlightDate'].nunique()
runway_capacity_per_hour = 40

hub_configs = {
    'DFW': {'df': hourly_wait_dfw, 'runways': 3},
    'ORD': {'df': hourly_wait_ord, 'runways': 3},
    'ATL': {'df': hourly_wait_atl, 'runways': 5} # ATL has 5 primary departure configuration options typically
}

for hub_name, config in hub_configs.items():
    df = config['df']
    runways = config['runways']
    
    peak_row = df.loc[df['ArrivalRate'].idxmax()]
    lambda_peak = peak_row['ArrivalRate'] / num_days
    avg_wait_peak = peak_row['AvgWaitTime']
    rho = lambda_peak / (runways * runway_capacity_per_hour)
    
    summary_data.append({
        'Hub': hub_name,
        'PeakWaitTime': avg_wait_peak,
        'TrafficIntensity': rho,
        'PeakArrivalRate': lambda_peak
    })

summary_df = pd.DataFrame(summary_data)

# comparartive bar chart for peak wait times across the three hubs
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    summary_df['Hub'], 
    summary_df['PeakWaitTime'], 
    color=['#D35400', '#1B4F72', '#27AE60'], 
    width=0.5,
    alpha=0.85
)

ax.set_title('Comparative Peak Runway Wait Time ($W$) Across Major Hubs', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Airport Hub', fontsize=12, labelpad=10)
ax.set_ylabel('Peak Average Wait Time (Minutes)', fontsize=12, labelpad=10)
ax.grid(False)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.1f} min',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5), 
                textcoords="offset points",
                ha='center', va='bottom', fontweight='bold', fontsize=11)

sns.despine(left=False, bottom=False)
fig.tight_layout()
plt.show()

print("\n" + "=" * 55)
print("FINAL QUANTITATIVE QUEUING SUMMARY TABLE")
print("=" * 55)
print(summary_df.to_string(index=False))
print("=" * 55)

In [5]:
for hub_name, config in hub_configs.items():
    df = config['df']
    peak_row = df.loc[df['ArrivalRate'].idxmax()]
    print(f"{hub_name}: peak hour {int(peak_row['DepHour'])}:00, observed queue delay = {peak_row['AvgWaitTime']:.1f} min")

ATL: peak hour 8:00, observed queue delay = 6.3 min
DFW: peak hour 8:00, observed queue delay = 9.7 min
ORD: peak hour 8:00, observed queue delay = 11.9 min


In [6]:
conn = sqlite3.connect("flight_delays.db")
arr = pd.read_sql_query("""
    SELECT Dest AS Hub, FlightDate, CRSArrTime
    FROM flight_info
    WHERE Cancelled = 0 AND Diverted = 0 AND Dest IN ('DFW','ORD','ATL')
""", conn)
conn.close()
arr['Hour'] = (arr['CRSArrTime'] // 100).clip(upper=23)

dep = (flights.groupby(['Origin','FlightDate','DepHour']).size().rename('dep')
       .reset_index().rename(columns={'Origin':'Hub','DepHour':'Hour'}))
arrc = arr.groupby(['Hub','FlightDate','Hour']).size().rename('arr').reset_index()

ops = dep.merge(arrc, on=['Hub','FlightDate','Hour'], how='outer').fillna(0)
ops['total'] = ops['dep'] + ops['arr']   # all runway operations in that hour

mu = 40
rows = []
for hub, g in ops.groupby('Hub'):
    by_hour = g.groupby('Hour')['total']
    peak_hour = by_hour.mean().idxmax()
    lam_avg = by_hour.mean()[peak_hour]
    lam_busy = by_hour.quantile(0.90)[peak_hour]
    obs = hub_configs[hub]['df'].set_index('DepHour').loc[peak_hour, 'AvgWaitTime']
    for c in range(3, 7):
        rows.append({
            'Hub': hub, 'PeakHour': int(peak_hour), 'c': c,
            'rho_avg': round(lam_avg / (c * mu), 2),
            'rho_busy': round(lam_busy / (c * mu), 2),
            'model_wait_avg': round(mmc_wait(lam_avg, mu, c), 1),
            'model_wait_busy': round(mmc_wait(lam_busy, mu, c), 1),
            'observed_wait': round(obs, 1),
        })

pd.set_option('display.width', 200)
print(pd.DataFrame(rows).to_string(index=False))

Hub  PeakHour  c  rho_avg  rho_busy  model_wait_avg  model_wait_busy  observed_wait
ATL         8  3     1.20      1.32             inf              inf            6.3
ATL         8  4     0.90      0.99             2.8             42.0            6.3
ATL         8  5     0.72      0.79             0.4              0.8            6.3
ATL         8  6     0.60      0.66             0.1              0.2            6.3
DFW         8  3     1.14      1.22             inf              inf            9.7
DFW         8  4     0.85      0.91             1.8              3.5            9.7
DFW         8  5     0.68      0.73             0.3              0.5            9.7
DFW         8  6     0.57      0.61             0.1              0.1            9.7
ORD        18  3     1.25      1.40             inf              inf           16.4
ORD        18  4     0.94      1.05             5.1              inf           16.4
ORD        18  5     0.75      0.84             0.6              1.2        

In [7]:
c = 4
out = []
for hub, g in ops.groupby('Hub'):
    by_hour = g.groupby('Hour')['total']
    peak_hour = by_hour.mean().idxmax()
    lam = by_hour.mean()[peak_hour]
    base = mmc_wait(lam, mu, c)
    shift = mmc_wait(lam * 0.90, mu, c)
    out.append({'Hub': hub, 'lam': round(lam, 1), 'rho': round(lam / (c * mu), 2),
                'base': round(base, 1), 'shift10': round(shift, 1),
                'pct_cut': round(100 * (base - shift) / base)})
print(pd.DataFrame(out).to_string(index=False))

Hub   lam  rho  base  shift10  pct_cut
ATL 143.4 0.90   2.8      1.2       58
DFW 136.5 0.85   1.8      0.9       51
ORD 149.9 0.94   5.1      1.6       69
